# Аналитический отчёт по заявкам платформы «НОРД-Класс»

**Этап 1 итоговой контрольной работы.** Ноутбук читает `data/final/enrollments_raw.csv`,
проверяет строки теми же валидаторами, что и ETL-конвейер (Этапы 3 и 5), и считает
показатели для руководителя платформы.

* исходные файлы не изменяются;
* запуск: **Kernel → Restart & Run All**;
* сторонние библиотеки не нужны — только стандартная библиотека Python;
* все числа в выводах совпадают с контрольными числами `etl_report.txt`.

## 1. Загрузка и входная проверка

Читаем «сырой» CSV через `csv.DictReader` и прогоняем каждую строку через
`etl_import.validate_source()`. Функция возвращает две коллекции: принятые
уникальные заявки и отклонённые строки с кодами ошибок. Дубликат по
`enrollment_id` попадает в отклонённые — в аналитику он не должен попасть дважды.

In [1]:
import csv
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import etl_import

DATA_DIR = Path("data/final")

students = etl_import.known_students(DATA_DIR / "students.csv")
courses = etl_import.known_courses(DATA_DIR / "courses.csv")
columns, raw_rows = etl_import.read_source(DATA_DIR / "enrollments_raw.csv")
accepted, rejected = etl_import.validate_source(raw_rows, students, courses)

valid_rows = [row for _, row in accepted]

print("Колонки CSV:              ", ", ".join(columns))
print("Студентов в справочнике:  ", len(students))
print("Курсов в справочнике:     ", len(courses))
print("Прочитано строк:          ", len(raw_rows))
print("Принято уникальных заявок:", len(valid_rows))
print("Отклонено строк:          ", len(rejected))

Колонки CSV:               enrollment_id, student_email, course_code, status, created_date, duration_hours, format
Студентов в справочнике:   16
Курсов в справочнике:      5
Прочитано строк:           19
Принято уникальных заявок: 10
Отклонено строк:           9


**Пояснение.** В файле 19 строк. После проверки осталось 10 уникальных корректных
заявок: 8 строк содержат ошибки данных, ещё одна строка — повтор уже принятой
заявки `EDU-202610-0006`. Одна строка (`EDU-202610-0017`) содержит
неэкранированную запятую в e-mail, поэтому её поля сдвинуты — она отклонена
целиком с причиной `malformed_row`, а не разобрана «по кусочкам».

## 2. Показатель 1. Распределение заявок по статусам

`Counter` считает частоты одним проходом. Полоса из символов `#` заменяет
график: ноутбук остаётся воспроизводимым без matplotlib.

In [2]:
def bar(value: int, maximum: int, width: int = 28) -> str:
    """Текстовая полоса: длина пропорциональна значению."""
    filled = round(width * value / maximum) if maximum else 0
    return "#" * filled + "." * (width - filled)


status_counts = Counter(row["status"] for row in valid_rows)
print(f"{'статус':<12}{'заявок':>7}  {'доля':>7}  полоса")
for status, count in status_counts.most_common():
    share = count / len(valid_rows)
    print(f"{status:<12}{count:>7}  {share:>7.1%}  "
          f"{bar(count, max(status_counts.values()))}")
print("Итого:", sum(status_counts.values()))

статус       заявок     доля  полоса
enrolled          4    40.0%  ############################
on_review         3    30.0%  #####################.......
completed         2    20.0%  ##############..............
cancelled         1    10.0%  #######.....................
Итого: 10


**Интерпретация.** Больше всего заявок в статусе `enrolled` — 4 из 10 (40%):
они оплачены и находятся в обучении. Ещё 3 заявки (30%) ждут решения оператора
(`on_review`), 2 завершены (`completed`), 1 отменена. Ни одной заявки в статусе
`new` не осталось — первичная обработка не создаёт очередь.

## 3. Показатель 2. Нагрузка по курсам

Сопоставляем код курса с названием из справочника `courses.csv`.

In [3]:
with (DATA_DIR / "courses.csv").open(encoding="utf-8-sig", newline="") as source:
    course_rows = list(csv.DictReader(source))

course_titles = {row["course_code"]: row["title"] for row in course_rows}
course_prices = {row["course_code"]: int(row["price_rub"]) for row in course_rows}

course_counts = Counter(row["course_code"] for row in valid_rows)
print(f"{'код':<8}{'курс':<22}{'заявок':>7}")
for code, count in course_counts.most_common():
    print(f"{code:<8}{course_titles[code]:<22}{count:>7}")
print(f"{'SE-250':<8}{course_titles['SE-250']:<22}{course_counts.get('SE-250', 0):>7}")

код     курс                   заявок
PY-100  Основы Python               5
DA-200  Анализ данных               2
WD-300  Веб-разработка              2
QA-150  Тестирование ПО             1
SE-250  Архитектура ПО              0


**Интерпретация.** Половина заявок (5 из 10) приходится на «Основы Python» —
курс-локомотив. «Анализ данных» и «Веб-разработка» дали по 2 заявки,
«Тестирование ПО» — одну. Курс «Архитектура ПО» (`SE-250`) не получил ни одной
заявки: его стоит либо активнее продвигать, либо пересмотреть расписание.

## 4. Показатель 3. Выручка по курсам

Выручку считаем только по оплаченным статусам `enrolled` и `completed`:
`new`, `on_review` и `cancelled` денег не приносят. Цена берётся из справочника
курсов, а не из файла заявок.

In [4]:
PAID_STATUSES = ("enrolled", "completed")
revenue = Counter()
paid_count = Counter()
for row in valid_rows:
    if row["status"] in PAID_STATUSES:
        revenue[row["course_code"]] += course_prices[row["course_code"]]
        paid_count[row["course_code"]] += 1

total_revenue = sum(revenue.values())
print(f"{'код':<8}{'оплачено':>9}{'выручка, руб.':>15}{'доля':>8}")
for code, amount in revenue.most_common():
    print(f"{code:<8}{paid_count[code]:>9}{amount:>15}{amount / total_revenue:>8.1%}")
print(f"{'ИТОГО':<8}{sum(paid_count.values()):>9}{total_revenue:>15}{1:>8.0%}")

код      оплачено  выручка, руб.    доля
PY-100          4         180000   54.2%
WD-300          1          84000   25.3%
DA-200          1          68000   20.5%
ИТОГО           6         332000    100%


**Интерпретация.** Оплачено 6 заявок на **332 000 руб.** Основной вклад даёт
Python (180 000 руб., 54,2%), затем веб-разработка (84 000 руб., 25,3%) и
анализ данных (68 000 руб., 20,5%). Важно: количество заявок и выручка
распределены по-разному — у Python дешёвый курс, но много заявок, а у
веб-разработки одна заявка даёт четверть выручки.

## 5. Показатель 4. Длительность обучения

Средняя и медиана считаются по принятым заявкам; `statistics.mean` и
`statistics.median` входят в стандартную библиотеку.

In [5]:
durations = [row["duration_hours"] for row in valid_rows]
print("Средняя длительность: ", statistics.mean(durations), "ч")
print("Медиана:              ", statistics.median(durations), "ч")
print("Минимум / максимум:   ", min(durations), "/", max(durations), "ч")

by_course = defaultdict(list)
for row in valid_rows:
    by_course[row["course_code"]].append(row["duration_hours"])

print(f"\n{'код':<8}{'заявок':>7}{'среднее':>9}{'медиана':>9}")
for code in sorted(by_course):
    values = by_course[code]
    print(f"{code:<8}{len(values):>7}{statistics.mean(values):>9.1f}"
          f"{statistics.median(values):>9.1f}")

Средняя длительность:  42.0 ч
Медиана:               36.0 ч
Минимум / максимум:    24.0 / 60.0 ч

код      заявок  среднее  медиана
DA-200        2     48.0     48.0
PY-100        5     36.0     36.0
QA-150        1     24.0     24.0
WD-300        2     60.0     60.0


**Интерпретация.** Средняя длительность — 42 часа, медиана — 36: типичная
заявка короче средней, потому что длинные курсы тянут среднее вверх. Самая
большая нагрузка у веб-разработки (60 ч), самая компактная — у тестирования
(24 ч). Для планирования преподавателей лучше опираться на медиану, а для
фонда оплаты труда — на сумму часов.

## 6. Показатель 5. Динамика заявок по месяцам

Период берём из даты создания: первые семь символов `ГГГГ-ММ` дают месяц.

In [6]:
month_counts = Counter(row["created_date"][:7] for row in valid_rows)
print(f"{'месяц':<9}{'заявок':>7}")
for month in sorted(month_counts):
    print(f"{month:<9}{month_counts[month]:>7}")

месяц     заявок
2026-09        3
2026-10        7


**Интерпретация.** В сентябре 3 заявки, в октябре 7 — рост более чем в два раза.
Сентябрь — старт набора, октябрь — основной поток. Если тренд сохранится,
к ноябрю стоит заранее расширить группу поддержки.

## 7. Показатель 6. Форматы обучения

Сравниваем онлайн и офлайн, чтобы понять потребность в аудиториях.

In [7]:
format_counts = Counter(row["format"] for row in valid_rows)
print(f"{'формат':<9}{'заявок':>7}  {'доля':>7}  полоса")
for name, count in format_counts.most_common():
    print(f"{name:<9}{count:>7}  {count / len(valid_rows):>7.1%}  "
          f"{bar(count, max(format_counts.values()))}")

формат    заявок     доля  полоса
online         6    60.0%  ############################
offline        4    40.0%  ###################.........


**Интерпретация.** 6 заявок из 10 (60%) — онлайн, 4 (40%) — офлайн.
Офлайн-формат требует аудиторий: две заявки на веб-разработку и две на анализ
данных. Онлайн-группы можно масштабировать без дополнительных помещений.

## 8. Показатель 7. Причины отклонения строк

Отклонённые строки — это не «мусор»: у каждой есть причина, которую можно
показать отправителю. `Counter.update()` суммирует коды ошибок; у одной строки
их может быть несколько.

In [8]:
reason_counts = Counter()
for _, _, errors in rejected:
    reason_counts.update(errors)

print(f"{'код ошибки':<30}{'раз':>5}")
for reason, count in reason_counts.most_common():
    print(f"{reason:<30}{count:>5}")
print("Всего отклонённых строк:", len(rejected))

код ошибки                      раз
invalid_enrollment_id             3
invalid_email                     1
unknown_student                   1
missing_field:student_email       1
unknown_course                    1
invalid_status                    1
invalid_duration                  1
malformed_row                     1
duplicate_enrollment_id           1
Всего отклонённых строк: 9


**Интерпретация.** Чаще всего встречается неверный идентификатор
(`invalid_enrollment_id`, 3 раза): строчная буква, короткий номер, буквы вместо
месяца. По одному разу встретились пустой e-mail, неизвестный курс, недопустимый
статус, отрицательная длительность, неэкранированная запятая и дубликат.
Одна строка может давать несколько причин, поэтому сумма кодов больше числа
отклонённых строк. Человекочитаемые формулировки собраны в
`validators.REASON_MESSAGES`.

## 9. Показатель 8. Вовлечённость студентов

Сравниваем справочник студентов с теми, у кого есть хотя бы одна принятая заявка.

In [9]:
with_active = {row["student_email"] for row in valid_rows}
without_enrollments = sorted(students - with_active)

print("Студентов с заявками:      ", len(with_active))
print("Студентов без заявок:      ", len(without_enrollments))
print("Доля вовлечённых студентов:", f"{len(with_active) / len(students):.1%}")
for email in without_enrollments:
    print("  ", email)

Студентов с заявками:       10
Студентов без заявок:       6
Доля вовлечённых студентов: 62.5%
   belova@example.com
   gusev@example.com
   morozova@example.com
   pavlova@example.com
   titova@example.com
   voronov@example.com


**Интерпретация.** Заявки есть у 10 из 16 студентов (62,5%). Шесть человек
зарегистрированы, но ещё не выбрали курс — это готовая аудитория для рассылки.
Отдельно стоит проверить, не потерялись ли их заявки среди отклонённых строк.

## 10. Сводка ключевых чисел

In [10]:
summary = {
    "строк прочитано": len(raw_rows),
    "принято заявок": len(valid_rows),
    "отклонено строк": len(rejected),
    "заявок в работе (new + on_review)": status_counts["new"] + status_counts["on_review"],
    "курсов с заявками": len(course_counts),
    "оплачено заявок": sum(paid_count.values()),
    "выручка, руб.": total_revenue,
    "средняя длительность, ч": round(statistics.mean(durations), 1),
}
for name, value in summary.items():
    print(f"{name:<36}{value}")

строк прочитано                     19
принято заявок                      10
отклонено строк                     9
заявок в работе (new + on_review)   3
курсов с заявками                   4
оплачено заявок                     6
выручка, руб.                       332000
средняя длительность, ч             42.0


## 11. Выводы для руководителя

1. **Воронка работает, но есть «застрявшие» заявки.** 4 заявки уже enrolled,
   2 завершены, 3 находятся на проверке. Очередь `on_review` — 30% потока;
   если её не разгружать, время до зачисления вырастет.
2. **Качество входных данных — главный риск.** Из 19 строк автоматически
   принято только 10; 9 отклонены, и у каждой есть причина. Самая частая
   ошибка — идентификатор заявки (3 случая). Нужна маска ввода на стороне
   источника заявок.
3. **Python — локомотив по количеству, веб-разработка — по выручке.**
   5 заявок на Python дают 54% потока, но 84 000 руб. с одной заявки на
   веб-разработку — это 25% выручки. Оба курса стратегически важны.
4. **Онлайн преобладает (60%).** Офлайн-группы нужно планировать под
   веб-разработку и анализ данных; остальное можно вести онлайн.
5. **Курс SE-250 не востребован** — ни одной заявки. Требуется решение:
   продвижение или замена программы.
6. **Рост потока в октябре** (3 → 7 заявок) требует заранее спланировать
   ресурсы поддержки на ноябрь.

## 12. Контроль воспроизводимости

Последняя ячейка проверяет, что при повторном запуске ноутбук получает те же
числа, что и ETL-конвейер. Если данные изменятся, проверка упадёт — это защита
от «тихого» расхождения отчёта и базы.

In [11]:
assert len(raw_rows) == 19
assert len(valid_rows) == 10
assert len(rejected) == 9
assert status_counts["enrolled"] == 4
assert sum(status_counts.values()) == len(valid_rows)
assert total_revenue == 332_000
assert statistics.mean(durations) == 42.0
print("Все контрольные числа совпали — отчёт воспроизводим.")

Все контрольные числа совпали — отчёт воспроизводим.


## 13. Как воспроизвести

1. Открыть папку `edu_pipeline` (в ней лежит этот ноутбук).
2. Запустить `Kernel → Restart & Run All`.
3. Убедиться, что последняя ячейка вывела «Все контрольные числа совпали».

Ноутбук использует `etl_import` и `validators` из той же папки, поэтому
проверка данных здесь не дублируется, а совпадает с ETL по построению.